# Retraining Cellpose on Custom Data

In [ ]:
# /// script
# requires-python = ">=3.12"
# dependencies = [
#     "matplotlib",
#     "cellpose",
# ]
# ///

## <mark style="color: black; background-color: rgb(127,196,125); padding: 3px; border-radius: 5px;">Overview</mark>

[Website](https://www.cellpose.org) | [GitHub](https://github.com/mouseland/cellpose) | [Paper](https://www.biorxiv.org/content/10.1101/2025.04.28.651001v1) | [Cellpose Documentation](https://cellpose.readthedocs.io/en/latest/index.html) | [Cellpose API](https://cellpose.readthedocs.io/en/latest/api.html#)

In this section, we will walk through how to **retrain Cellpose on your own data**. This is useful when the default models don’t perform well on your specific cell type, staining method, or imaging modality.

Retraining allows Cellpose to learn directly from your examples, leading to better segmentation accuracy and more relevant masks for your experiments.

To go through the training process, you need **pairs of raw microscopy images and their corresponding label masks**. The raw images are what you want to segment, and the label masks are the ground truth segmentations that the model will learn from.

<p class="alert alert-warning">
    <strong>⚠️ Note:</strong> Cellpose runs significantly faster on a GPU. It supports both NVIDIA GPUs (CUDA) and Apple Silicon (MPS). If you don't have either, we recommend running this notebook on <a href="https://colab.research.google.com/github/bobiac/bobiac-book/blob/gh-pages/colab_notebooks/05_segmentation/deep_learning/cellpose_retraining_colab.ipynb" target="_blank"> Google Colab</a> for faster performance.
</p>

:::{dropdown} NVIDIA GPU (CUDA - Windows/Linux)

In order to use Cellpose in this notebook with an NVIDIA GPU:
1. you need to have the [NVIDIA drivers](https://www.nvidia.com/en-us/drivers/) installed on your system.
2. you can run `nvidia-smi` in the terminal to check your CUDA version (shown in the top-right of the output, e.g. `CUDA Version: 13.0.0`).
3. update the `# /// script` block at the top of this notebook to install the appropriate version of [PyTorch with CUDA support](https://pytorch.org/get-started/locally/) (replace `cu130` with your CUDA version):

```python
    # /// script
    # requires-python = ">=3.12"
    # dependencies = [
    #     "matplotlib",
    #     "cellpose",
    #     "torch",
    #     "torchvision",
    # ]
    #
    # [tool.uv.sources]
    # torch = { index = "pytorch-cu130" }
    # torchvision = { index = "pytorch-cu130" }
    #
    # [[tool.uv.index]]
    # name = "pytorch-cu130"
    # url = "https://download.pytorch.org/whl/cu130"
    # explicit = true
    # ///
```

4. re-run the notebook using `uvx juv run`.
:::

## <mark style="color: black; background-color: rgb(127,196,125); padding: 3px; border-radius: 5px;">How to Retrain Cellpose</mark>

The retraining process can be performed using the [Cellpose GUI](https://cellpose.readthedocs.io/en/latest/gui.html#training-your-own-cellpose-model), which lets you train a model interactively, or using the code-based approach shown in this tutorial.

:::Dropdown title="Run the Cellpose GUI"
If using `uv`, you can simply start `Cellpose` by running in your terminal:

`uvx "cellpose[gui]"`

This will open the Cellpose GUI.

If you have an NVIDIA GPU and want to run the GUI with CUDA support, you can install `torch` with CUDA from the appropriate index (replace `cu130` with your CUDA version). Note the `--index-strategy unsafe-best-match` flag, which is needed so that `uv` resolves `cellpose` itself from PyPI rather than from the PyTorch index:

`uvx --index-strategy unsafe-best-match --with torch --with torchvision --index https://download.pytorch.org/whl/cu130 "cellpose[gui]"`

:::

For this tutorial, we will use the sample dataset provided by Cellpose, which includes both training and test images pairs. You can download the dataset here:
<a href="../../../_static/data/05_segmentation_cellpose_training.zip" download>
<i class="fas fa-download"></i> Cellpose Training Dataset</a>. 

<p class="alert alert-info">
    <strong>🚧 Note:</strong> We will use only the sample dataset provided by Cellpose to demonstrate the retraining pipeline. If you want to retrain Cellpose on your own data, you will probably need more image pairs for training and for testing, as well as some validation data.
</p>

## <mark style="color: black; background-color: rgb(127,196,125); padding: 3px; border-radius: 5px;">Creating Label Masks</mark>

As mentioned above, the dataset we will use already includes label masks, but if you want to retrain Cellpose on your own data, you will need to create these masks yourself.

There are different tools available. One example is the Cellpose GUI itself, which lets you modify and save updated labels in a user-friendly way. You can see the [Cellpose documentation](https://cellpose.readthedocs.io/en/latest/gui.html#training-your-own-cellpose-model) for more details on how to create and edit label masks.

Another option is to use annotation tools like [napari](https://napari.org/). If using `uv`, you can simply start `napari` by running in your terminal:

`uvx "napari[all]"`

The `napari` GUI will open and you can load your raw images and create or modify and then save the corresponding label masks using the annotation tools available.

## <mark style="color: black; background-color: rgb(127,196,125); padding: 3px; border-radius: 5px;">Retraining Pipeline</mark>

The retraining pipeline consists of the following steps:
1. **Data Preparation**: Organize your raw images and label masks into a format that Cellpose can use for training.

2. **Model Configuration**: Set up the training parameters, such as the number of epochs, learning rate, and batch size.

3. **Run the standard model on test data [optional]**: Before training, in this tutorial we will run the standard Cellpose model on the test data to see how it performs before retraining.

4. **Train the Model**: Use the prepared data to train a new Cellpose model.

5. **Evaluate the Model**: After training, evaluate the performance of the new model on the test data and compare it to the performance of the standard model.


## <mark style="color: black; background-color: rgb(127,196,125); padding: 3px; border-radius: 5px;">Import Libraries</mark>

## <mark style="color: black; background-color: rgb(127,196,125); padding: 3px; border-radius: 5px;">Setup</mark>

## <mark style="color: black; background-color: rgb(127,196,125); padding: 3px; border-radius: 5px;">Data Handling</mark>

Cellpose expects images and their corresponding masks to live **in the same folder**. The file names must share the same prefix and only differ by the suffix. For example, if the raw image is named `img_0.tif`, the corresponding mask should be named `img_0_seg.tif` (`_seg` here is just an example, you can use any suffix you like).

You'll also need to **split your data** into a `train` and a `test` folder. The model learns from the training set and is evaluated on the test set (images it has never seen during training).

```
cellpose_data/
├── train/
│   ├── img_0.tif
│   ├── img_0_seg.tif
│   ├── img_1.tif
│   ├── img_1_seg.tif
│   └── ...
└── test/
    ├── img_8.tif
    ├── img_8_seg.tif
    ├── img_9.tif
    ├── img_9_seg.tif
    └── ...
```

After organizing your data, the first step is to define the `train` and `test` directories, then load the data with [`io.load_train_test_data`](https://cellpose.readthedocs.io/en/latest/api.html#cellpose.io.load_train_test_data), passing the mask (and image if needed) suffixes so Cellpose can pair them correctly.

## <mark style="color: black; background-color: rgb(127,196,125); padding: 3px; border-radius: 5px;">Init the Model</mark>

The next step after organizing and loading your data is to initialize the Cellpose model you want to retrain.

## <mark style="color: black; background-color: rgb(127,196,125); padding: 3px; border-radius: 5px;">How does the standard model perform on the test data? (Optional)</mark>

Before training a new model, let's see how the standard Cellpose model performs on the test data.

Now we'll quantify how well the standard model segments the test images by comparing the predicted masks to the ground truth labels using [`metrics.average_precision`](https://cellpose.readthedocs.io/en/latest/api.html#cellpose.metrics.average_precision).

For each test image, every predicted mask is matched to a ground truth mask based on their [Intersection over Union (IoU)](https://en.wikipedia.org/wiki/Jaccard_index): the overlap area divided by the union area of the two masks.

<div align="center"> <img src="https://raw.githubusercontent.com/bobiac/bobiac-book/main/_static/images/cellpose/iou.png" alt="iou" width="300"></div>

A predicted mask counts as a **true positive** (TP) if its IoU with a ground truth mask is above a given threshold; otherwise it is a **false positive** (FP), and any unmatched ground truth mask is a **false negative** (FN). The average precision (AP) is then computed as:

$$
AP = \frac{TP}{TP + FP + FN}
$$

By default, `metrics.average_precision` computes this at IoU thresholds of **0.5**, **0.75**, and **0.9**. A higher threshold requires a tighter overlap between predicted and ground truth masks, so AP typically decreases as the threshold increases. Comparing these values before and after retraining gives us a quantitative measure of how much the model improves on our data.

Let's now visualize the results for the test images.

## <mark style="color: black; background-color: rgb(127,196,125); padding: 3px; border-radius: 5px;">Train New Model</mark>

Now we're ready to retrain Cellpose using the `train_seg` method from the `train` module.

Here we will change few training parameters but you can find the full parameters description for the `train.train_seg` method in the dropdown below or in the [Cellpose API documentation](https://cellpose.readthedocs.io/en/latest/api.html#module-cellpose.train)

:::{dropdown} CellposeSam train.train_seg() Parameters

**Input Data**

| Parameter | Default | What it does |
|---|---|---|
| `train_data` | `None` | List of numpy arrays (2D or 3D images). Mutually exclusive with `train_files`, use one or the other. |
| `train_labels` | `None` | List of integer label arrays matching `train_data`. `0` = bzackground, `1, 2, ...` = individual masks. |
| `train_files` | `None` | File paths to training images. Used instead of `train_data` when loading from disk. Use together with `load_files=True` (the default), this is the standard path when working from disk. |
| `train_labels_files` | `None` | File paths to the corresponding label files. |
| `test_data` / `test_labels` | `None` | Same as above but for validation. Used only to report test loss, does not affect gradient updates. |
| `test_files` / `test_labels_files` | `None` | Same as above, file-based version. |
| `load_files` | `True` | If `True`, loads images/labels from the `*_files` paths at the start of training. Set to `False` if you've already loaded them into arrays and don't want redundant I/O. |
| `channel_axis` | `None` | Which axis in the arrays is the channel axis. `None` = infer automatically. |

**Sampling & Epoch Control**

| Parameter | Default | What it does |
|---|---|---|
| `train_probs` | `None` | Per-image sampling probability for each epoch. If `None`, all images get equal weight (`1/n`). The right lever if you want to hard-mine difficult images or balance an uneven dataset. Must sum to 1 after normalization. |
| `test_probs` | `None` | Same for test images. |
| `nimg_per_epoch` | `None` | How many images (with random augmentation each time) Cellpose samples per epoch, doing one gradient update per image. If `None`, defaults to `len(train_data)`. For a large dataset (50/100+ images), leave as `None`, one pass per epoch is already enough gradient steps. For a small dataset (5-10 images), set it higher than your dataset size (e.g. `50` or `100` for 5 images): each image gets sampled multiple times per epoch with different random augmentations (crop, rotation, flip, scale), and more updates/epoch help the linear LR warmup over the first 10 epochs actually move the weights.
| `nimg_test_per_epoch` | `None` | Same as `nimg_per_epoch` but for test loss reporting only, it does not affect training. For a small test set (2-5 images), leave as `None` (uses all) for a stable loss estimate, subsampling a tiny test set gives noisy numbers. For a large test set (100+ images), set it lower (e.g. `20`) to speed up per-epoch evaluation at the cost of noisier test loss curves. |
| `min_train_masks` | `5` | Images with fewer than this many mask instances are silently dropped before training starts, guarding against nearly-empty label images corrupting training. If your dataset is small, check how many images actually survive this filter. |

**Optimizer**

| Parameter | Default | What it does |
|---|---|---|
| `learning_rate` | `1e-5` | learning rate (LR) for the AdamW algorithm used by Cellpose. Controls how large each weight update is. Too high → the model overshoots and training becomes unstable (loss oscillates). Too low → the model learns very slowly or gets stuck. For fine-tuning a pre-trained model like Cellpose, a small value is preferred to avoid erasing what the model already knows (e.g. `1e-5` or 0.00001). Note that `learning_rate` is also ramped up linearly from 0 to its target value over the first 10 epochs (regardless of `n_epochs`), so with very few epochs the target learning rate may never be reached. |
| `weight_decay` | `0.1` | AdamW algorithm weight decay (L2 regularization). `0.1` is the modern default from the AdamW paper. |
| `SGD` | `False` | Deprecated as of v4.0.1. AdamW is always used regardless of this value. |
| `n_epochs` | `100` | one epoch = one full pass through all training images. More epochs give the model more time to learn, but too many can lead to *overfitting*: the model memorizes the training data and performs worse on new images. Watch the test loss, if it starts rising while the train loss keeps falling, you've trained too long. |
| `batch_size` | `1` | number of `bsize`x`bsize` pixels image tiles processed simultaneously on the GPU before the model updates its weights (the image is split into tiles before being processed). Larger batches give more stable gradient estimates but require more memory. Smaller batches are noisier but work well for small datasets and use less GPU memory. See the [cellpose_notebook](.//cellpose_notebook.ipynb) for more details on batch size and GPU performance.|
| `bsize` | `256` | Size of each tile in pixels (`bsize × bsize`). Fixed to 256x256 pixels for the SAM backbone (`cpsam`), don't override it. |

**Augmentation & Preprocessing**

| Parameter | Default | What it does |
|---|---|---|
| `normalize` | `True` | If `True`, applies default percentile normalization per image. If a dict, merges with default normalize params (e.g. `{"tile_norm_blocksize": 128}` for tile-based normalization on large images). |
| `compute_flows` | `False` | If `True`, recomputes flow fields from label masks during training rather than using cached flows. Slower but ensures flows match labels exactly if labels were recently edited. |
| `rescale` | `False` | If `True`, rescales each image during training based on its object diameter relative to the model's expected diameter (`net.diam_mean`). Helps when objects in your data differ significantly in size from the training distribution. |
| `scale_range` | `None` | Range of random scale augmentation passed to `random_rotate_and_resize()`. Default resolves to `0.5`, meaning scale is sampled in `[1-0.5, 1+0.5]`. Larger values = more aggressive scale augmentation. |

**Loss**

| Parameter | Default | What it does |
|---|---|---|
| `class_weights` | `None` | Array of per-class weights passed to `nn.CrossEntropyLoss(weight=...)`. Use this to up-weight underrepresented mask classes. Converted to a float32 CUDA tensor internally. |

**Saving**

| Parameter | Default | What it does |
|---|---|---|
| `save_path` | `None` | Directory where the trained model is written. If `None`, no model is saved to disk. |
| `save_every` | `100` | Save a checkpoint every N epochs. |
| `save_each` | `False` | If `True`, each checkpoint gets a unique filename (epoch-stamped). If `False`, each save overwrites the previous checkpoint. |
| `model_name` | `None` | The filename stem for the saved model. If `None`, an automatic name is generated. |

:::

We can also plot the training and test losses over epochs to see how the model is learning.

## <mark style="color: black; background-color: rgb(127,196,125); padding: 3px; border-radius: 5px;">Evaluate on test data</mark>

To evaluate the new model, we can run it on the test images and compute the average precision again to see (if and) how much it improved compared to the standard model.